In [ ]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivasingh4945/student-social-media-and-mental-health-impact")

print("Path to dataset files:", path)

In [ ]:
import os

print(os.listdir(path))

In [ ]:
import pandas as pd

df = pd.read_csv(
    os.path.join(path, "Student Social Media And Mental Health Impact.csv")
)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['Mental_Health_Score'].value_counts()

In [ ]:
df.describe()

In [ ]:
import seaborn as sns
sns.histplot(df['Mental_Health_Score'],kde=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True)

In [ ]:
num_features=df.select_dtypes(include='number')

In [ ]:
num_features

In [ ]:
q1=num_features.quantile(0.25)
q3=num_features.quantile(0.75)
iqr=q3-q1
lower=q1-1.5*iqr
upper=q3+1.5*iqr
outliers=(num_features<lower)|(num_features>upper)

In [ ]:
print(outliers['Physical_Activity_Hours'].values)

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
df['Country'].value_counts().head(11)

In [ ]:
countries=['India','USA','Canada','Australia','UK','Germany','Mexico','Turkey','France','Spain']


In [ ]:
def country_name(country):
  if country in countries:
    return country
  else:
    return 'Other'


In [ ]:
df['Country']=df['Country'].apply(country_name)

In [ ]:
df['Country'].value_counts()

In [ ]:
num_coln=df.select_dtypes(include='number').columns

In [ ]:
num_coln

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split




skwed_column=['Study_Hours']
other_num_coln=['Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks','Physical_Activity_Hours', 'Sleep_Hours_Per_Night']


ordinal=['Stress_Level']

one_hot=['Gender','Country','Academic_Level','Most_Used_Platform','Purpose_Of_Use']

featured_coln=skwed_column+other_num_coln+ordinal+one_hot

X=df[featured_coln]
y=df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,OneHotEncoder,FunctionTransformer
from sklearn.compose import ColumnTransformer


skew_pipeline=Pipeline(steps=[('log_transform',FunctionTransformer(np.log1p)),
                               ('scaler',StandardScaler())])

plain_num_pipeline=Pipeline(steps=[('scaler',StandardScaler())])

ordinal_pipeline=Pipeline(steps=[('encode',OrdinalEncoder(categories=[['Low','Medium','High','Very High']]))])

one_hot_pipeline=Pipeline(steps=[('encode',OneHotEncoder(handle_unknown='ignore'))])





In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ('Skewed column',skew_pipeline,skwed_column),
    ('plain scaling',plain_num_pipeline,other_num_coln),
    ('ordinal encoding',ordinal_pipeline,ordinal),
    ('one hot encoding',one_hot_pipeline,one_hot)
])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error

lr_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
])

lr_pipeline.fit(X_train,y_train)
lr_pred=lr_pipeline.predict(X_test)

lr_mse=mean_squared_error(y_test,lr_pred)
lr_r2=r2_score(y_test,lr_pred)
lr_mae=mean_absolute_error(y_test,lr_pred)


In [ ]:
lr_mae


In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train,y_train)
rf_pred=rf_pipeline.predict(X_test)
rf_train_pred=rf_pipeline.predict(X_train)

rf_test_mse=mean_squared_error(y_test,rf_pred)
rf_test_r2=r2_score(y_test,rf_pred)
rf_train_mse=mean_squared_error(y_train,rf_train_pred)
rf_train_r2=r2_score(y_train,rf_train_pred)

In [ ]:
param_grid={
    'model__n_estimators':[100,200,300],
    'model__max_depth':[5,10,15],
    'model__min_samples_split':[2,5,10],
    'model__min_samples_leaf':[1,2,4]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
random_search=RandomizedSearchCV(estimator=rf_pipeline,param_distributions=param_grid,n_iter=15,cv=5,scoring='r2',random_state=42,n_jobs=-1)

In [ ]:
random_search.fit(X_train,y_train)

In [ ]:
random_search.best_params_

In [ ]:
final_model=random_search.best_estimator_

In [ ]:
final_model.fit(X_train,y_train)

In [ ]:
y_final_test_pred=final_model.predict(X_test)
y_final_train_pred=final_model.predict(X_train)
mae_final=mean_absolute_error(y_test,y_final_test_pred)
r2_final_test=r2_score(y_test,y_final_test_pred)
r2_final_train=r2_score(y_train,y_final_train_pred)

In [ ]:
print(mae_final)
print(r2_final_test)
print(r2_final_train)

In [ ]:
import joblib
joblib.dump(final_model, 'MENTAL_HEALTH_MODEL.pkl')